In [1]:
#This is program for count E2E research

In [1]:
import pandas as pd
import arxiv

In [4]:
results = client.results(search)
def convert_to_str(vals):  # 文字列に変換
    if isinstance(vals, list):
        return list(map(str, vals))
    elif vals is None:
        return vals
    else:
        return str(vals)
        
def create_df(results):
    res0 = []
    for res in results:
        res_dict = dict(
            entry_id = res.entry_id,
            pdf_url = res.pdf_url,
            updated = res.updated, 
            published = res.published, 
            title = res.title, 
            authors = convert_to_str(res.authors),  # 文字列に変換
            summary = res.summary,
            comment = res.comment,
            journal_ref = res.journal_ref,
            doi = res.doi,
            primary_category = res.primary_category,
            categories = res.categories,
            links = convert_to_str(res.links),  # 文字列に変換
        )
        res0.append(res_dict)
    df = pd.DataFrame(res0)
    ndata = len(df)
    return df

df = create_df(results)


In [10]:
client = arxiv.Client(
    page_size = 100,
    delay_seconds = 3.0,
    num_retries = 10,      # 間欠障害が前提なので回数を確保する
)
search = arxiv.Search(
    query = 'abs:"autonomous driving" AND (abs:"E2E" OR abs:"End to End" OR "End-to-End")',
    max_results = 1000,
    sort_by = arxiv.SortCriterion.SubmittedDate,
    sort_order = arxiv.SortOrder.Descending,
)
df = create_df(client.results(search))
print(len(df))
query = 'abs:"autonomous driving" AND (abs:"E2E" OR abs:"End to End" OR "End-to-End")',

1000


In [11]:
import urllib.parse, urllib.request
import xml.etree.ElementTree as ET

NS = {'opensearch': 'http://a9.com/-/spec/opensearch/1.1/'}

def arxiv_total(query: str) -> int:
    url = ('http://export.arxiv.org/api/query?search_query='
           + urllib.parse.quote(query) + '&max_results=1')
    with urllib.request.urlopen(url) as r:
        root = ET.fromstring(r.read())
    return int(root.find('opensearch:totalResults', NS).text)

print(arxiv_total(query))

TypeError: quote_from_bytes() expected bytes

In [12]:
import numpy as np

def create_url(aid: str):
    apath = aid.split("/")[-1].split("v")[0]
    arxiv_id = f'ARXIV:{apath}'
    search_url = f'{semantic_base_url}/{arxiv_id}' + '?fields=' + fields 
    return arxiv_id

semantic_base_url = 'https://api.semanticscholar.org/graph/v1/paper'  # Semantic ScholarのベースURL
fields = 'citationCount,influentialCitationCount,referenceCount'  # サイトから取得する追加項
df['arxiv_id'] = df['entry_id'].apply(create_url)
tcol = fields.split(",") 
df[tcol] = np.nan  # 初期化

In [13]:
import requests
import time

HEADERS = {}   # APIキー取得後は {"x-api-key": API_KEY}

def post_batch(batch, max_retry=6):
    for attempt in range(max_retry):
        r = requests.post(semantic_base_url + '/batch',
                          params={'fields': fields}, json={"ids": batch},
                          headers=HEADERS, timeout=30)
        if r.status_code == 200:
            return r.json()
        if r.status_code in (429, 500, 502, 503, 504):
            wait = int(r.headers.get('Retry-After', 0)) or 5 * 2 ** attempt
            print(f'  {r.status_code}: retry in {wait}s (attempt {attempt+1})')
            time.sleep(wait)
            continue
        r.raise_for_status()
    return None


def get_citation_batch(df, start=0, batch_size=500):
    outs = []
    data = df['arxiv_id'].tolist()
    for i in range(start, len(data), batch_size):
        batch = data[i:i+batch_size]
        payload = post_batch(batch)
        if payload is None:
            print('Error: giving up. restart with start=', i)
            break
        rows = {pid: rec for pid, rec in zip(batch, payload) if rec is not None}
        if rows:
            outs.append(pd.DataFrame.from_dict(rows, orient="index"))
        time.sleep(5)
    if not outs:
        return df
    res = pd.concat(outs).rename_axis('arxiv_id').reset_index()
    return df.merge(res[['arxiv_id'] + tcol], on='arxiv_id', how='left')


start = 0  # 開始行
batch_size = 500  # バッチ数。500以下
df = get_citation_batch(df, start=start, batch_size=batch_size)

  429: retry in 5s (attempt 1)


In [14]:
df = df.drop(columns=[c + '_x' for c in tcol])
df = df.rename(columns={c + '_y': c for c in tcol})

In [15]:
df.sort_values(['influentialCitationCount'],ascending=False)

,entry_id,pdf_url,updated,published,title,authors,summary,comment,journal_ref,doi,primary_category,categories,links,arxiv_id,citationCount,influentialCitationCount,referenceCount
812,http://arxiv.org/abs/2303.12077v3,https://arxiv.org/pdf/2303.12077v3,2023-08-24 08:15:35+00:00,2023-03-21 17:59:22+00:00,VAD: Vectorized Scene Representation for Effic...,"[Bo Jiang, Shaoyu Chen, Qing Xu, Bencheng Liao...",Autonomous driving requires a comprehensive un...,Accepted to ICCV 2023. Code&Demos: https://git...,NaN,NaN,cs.RO,"[cs.RO, cs.CV]","[https://arxiv.org/abs/2303.12077v3, https://a...",ARXIV:2303.12077,739.0,141.0,52.0
853,http://arxiv.org/abs/2208.14437v2,https://arxiv.org/pdf/2208.14437v2,2023-01-30 02:39:49+00:00,2022-08-30 17:55:59+00:00,MapTR: Structured Modeling and Learning for On...,"[Bencheng Liao, Shaoyu Chen, Xinggang Wang, Ti...",High-definition (HD) map provides abundant and...,Accepted to ICLR 2023 as Spotlight Presentatio...,NaN,NaN,cs.CV,"[cs.CV, cs.RO]","[https://arxiv.org/abs/2208.14437v2, https://a...",ARXIV:2208.14437,419.0,111.0,44.0
873,http://arxiv.org/abs/2205.15997v1,https://arxiv.org/pdf/2205.15997v1,2022-05-31 17:57:19+00:00,2022-05-31 17:57:19+00:00,TransFuser: Imitation with Transformer-Based S...,"[Kashyap Chitta, Aditya Prakash, Bernhard Jaeg...",How should we integrate representations from c...,arXiv admin note: text overlap with arXiv:2104...,NaN,NaN,cs.CV,"[cs.CV, cs.AI, cs.LG, cs.RO]","[https://arxiv.org/abs/2205.15997v1, https://a...",ARXIV:2205.15997,762.0,110.0,134.0
726,http://arxiv.org/abs/2312.14150v3,https://arxiv.org/pdf/2312.14150v3,2025-01-16 10:57:44+00:00,2023-12-21 18:59:12+00:00,DriveLM: Driving with Graph Visual Question An...,"[Chonghao Sima, Katrin Renz, Kashyap Chitta, L...",We study how vision-language models (VLMs) tra...,Accepted to ECCV 2024 as Oral paper,NaN,NaN,cs.CV,[cs.CV],"[https://arxiv.org/abs/2312.14150v3, https://a...",ARXIV:2312.14150,609.0,86.0,101.0
645,http://arxiv.org/abs/2406.15349v2,https://arxiv.org/pdf/2406.15349v2,2024-10-31 17:58:34+00:00,2024-06-21 17:59:02+00:00,NAVSIM: Data-Driven Non-Reactive Autonomous Ve...,"[Daniel Dauner, Marcel Hallgarten, Tianyu Li, ...",Benchmarking vision-based driving policies is ...,NeurIPS 2024 Datasets and Benchmarks,NaN,NaN,cs.CV,"[cs.CV, cs.AI, cs.LG, cs.RO]","[https://arxiv.org/abs/2406.15349v2, https://a...",ARXIV:2406.15349,354.0,83.0,66.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
131,http://arxiv.org/abs/2604.18918v1,https://arxiv.org/pdf/2604.18918v1,2026-04-20 23:49:45+00:00,2026-04-20 23:49:45+00:00,From Particles to Perils: SVGD-Based Hazardous...,"[Linfeng Liang, Xiao Cheng, Tsong Yueh Chen, X...",Simulation-based testing of autonomous driving...,NaN,Proceedings of the ACM on Software Engineering...,10.1145/3808153,cs.SE,"[cs.SE, cs.LG]","[https://arxiv.org/abs/2604.18918v1, https://a...",ARXIV:2604.18918,0.0,0.0,78.0
461,http://arxiv.org/abs/2505.15005v1,https://arxiv.org/pdf/2505.15005v1,2025-05-21 01:23:31+00:00,2025-05-21 01:23:31+00:00,UniSTPA: A Safety Analysis Framework for End-t...,"[Hongrui Kou, Zhouhang Lyu, Ziyu Wang, Cheng W...",As autonomous driving technology continues to ...,NaN,NaN,NaN,cs.RO,"[cs.RO, cs.SE]","[https://arxiv.org/abs/2505.15005v1, https://a...",ARXIV:2505.15005,1.0,0.0,23.0
0,http://arxiv.org/abs/2607.20988v1,https://arxiv.org/pdf/2607.20988v1,2026-07-23 07:11:41+00:00,2026-07-23 07:11:41+00:00,HyWorldVLA: A Vision-Language-Action Model wit...,"[Quanfu Yu, Xian Wu, Hao Xu, Liulong Ma]",Vision-Language-Action (VLA) models augmented ...,20 pages with 13 figures,NaN,NaN,cs.CV,"[cs.CV, cs.AI]","[https://arxiv.org/abs/2607.20988v1, https://a...",ARXIV:2607.20988,0.0,0.0,0.0
329,http://arxiv.org/abs/2510.00060v3,https://arxiv.org/pdf/2510.00060v3,2026-02-27 09:00:30+00:00,2025-09-29 05:14:18+00:00,Less is More: Lean yet Powerful Vision-Languag...,"[Sheng Yang, Tong Zhan, Guancheng Chen, Yanfen...","In this work, we reconceptualize autonomous dr...",NaN,NaN,N

In [16]:
NS = {'opensearch': 'http://a9.com/-/spec/opensearch/1.1/'}
for y in range(2015, 2027):
    q = f'abs:"autonomous driving" AND submittedDate:[{y}01010000 TO {y}12312359]'
    q = f'abs:"autonomous driving" AND (abs:"E2E" OR abs:"End to End" OR "End-to-End")AND submittedDate:[{y}01010000 TO {y}12312359]'

    print(y, arxiv_total(q))
    time.sleep(3)

2015 0
2016 5
2017 10
2018 23
2019 35
2020 46
2021 68
2022 78
2023 105
2024 171
2025 324
2026 227


In [17]:
print(df['published'].dt.year.value_counts().sort_index())

published
2020     27
2021     68
2022     78
2023    105
2024    171
2025    324
2026    227
Name: count, dtype: int64
